# 01 · Site data (NASA POWER)
For each published soiling-measurement site in `data/soiling_sites.csv`, download one year (2024) of daily:
- **rainfall** (`PRECTOTCORR`, mm/day)
- **solar irradiance** (`ALLSKY_SFC_SW_DWN`, kWh/m²/day)
- **aerosol optical depth at 550 nm** (`AOD_55`, NASA MERRA-2 via POWER)

No login needed. Output: `data/site_daily_2024.csv`.

In [ ]:
REPO = "https://raw.githubusercontent.com/ishjeetsingh/india-solar-soiling/main/"
import requests, pandas as pd, time

sites = pd.read_csv(REPO + "data/soiling_sites.csv")
sites = sites.dropna(subset=["lat", "lon"])          # sites without coordinates are skipped
sites

## Download function
NASA POWER returns JSON; `-999` means missing.

In [ ]:
def get_power(lat, lon, start="20240101", end="20241231"):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {"parameters": "PRECTOTCORR,ALLSKY_SFC_SW_DWN,AOD_55",
              "community": "RE", "latitude": lat, "longitude": lon,
              "start": start, "end": end, "format": "JSON"}
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["properties"]["parameter"])
    df.index = pd.to_datetime(df.index)
    df = df.replace(-999, pd.NA)
    return df.rename(columns={"PRECTOTCORR": "rain_mm",
                              "ALLSKY_SFC_SW_DWN": "irradiance_kwh_m2",
                              "AOD_55": "aod_merra2"})

In [ ]:
frames = []
for _, s in sites.iterrows():
    df = get_power(s.lat, s.lon)
    df["site"] = s.site
    frames.append(df)
    time.sleep(1)

daily = pd.concat(frames).rename_axis("date").reset_index()
daily.to_csv("site_daily_2024.csv", index=False)
daily.groupby("site")[["rain_mm", "irradiance_kwh_m2", "aod_merra2"]].mean().round(2)